<div style="
    padding: 32px 36px;
    border-radius: 16px;
    background: linear-gradient(135deg, #102A43 0%, #243B53 100%);
    color: #F7F4EC;
    margin-bottom: 24px;
">
    <div style="font-size: 13px; letter-spacing: 0.16em; text-transform: uppercase; color: #D4A44C; font-weight: 700;">
        ConflictLens · Article validation notebook
    </div>
    <h1 style="margin: 10px 0 8px 0; font-size: 38px; font-weight: 700; color: #FFFFFF;">
        Seven years out of thirty-seven. Half the toll.
    </h1>
    <p style="margin: 0; max-width: 860px; font-size: 17px; line-height: 1.55; color: #D9E2EC;">
        Lightweight, article-specific notebook that recalculates the figures and headline numbers used in the draft from the validated ConflictLens analysis-ready panel.
    </p>
    <div style="margin-top: 24px; padding-top: 16px; border-top: 1px solid rgba(255,255,255,0.20); font-size: 13px; color: #BCCCDC;">
        UCDP OV · GED event composition · annual concentration · contributor ≠ perpetrator
    </div>
</div>

## 0. Purpose and scope

This notebook is intentionally narrow. It does **not** rebuild the panel and does **not** reproduce the broad Notebook 03 analysis. Its purpose is to recalculate, qualify and export the figures and numbers used by the article **“Seven years out of thirty-seven. Half the toll.”**

**Input contract**

- Primary source: `02_analysis_unit_year_panel_analysis_ready.parquet`.
- Uncertainty source: `02_country_year_panel_staging.parquet`, used only for the GED low / best / high fatality estimates that underlie the OV annual totals.
- Analytical grain: `analysis_unit_id + year`.
- Main metric: `ucdp_ov_total_deaths_best_zf`.
- Period: 1989–2025.
- Universe: `analysis_conflict_universe == True` and `unit_exists_in_year == True`.

**Guardrails kept throughout the notebook**

1. The seven-year headline refers specifically to the **UCDP best estimate**. The low and high estimates are shown separately.
2. A country-year contributor is a statistical grouping, **not** a perpetrator attribution.
3. `0` and `NA` are not semantically identical; curated `_zf` metrics are used only after filtering out pre-existence rows.
4. Historical and legal conflict labels are not inferred from aggregate bars or event-count composition.
5. Intensity and diffusion are treated as distinct but potentially associated dimensions.
6. The notebook is descriptive: no causal claim, prediction or operational inference.
7. Israel 2025 is reported only as a raw statistical contributor and is deliberately excluded from GED-type categorisation charts.

## Navigation

1. [Environment and input](#environment-input)
2. [Annual curve, recent clustering and estimate sensitivity](#annual-curve)
3. [Country-year concentration](#country-year-concentration)
4. [Peak-year contributor structures and Israel 2025](#peak-contributors)
5. [GED event composition for selected peaks](#ged-composition)
6. [Intensity versus diffusion](#intensity-diffusion)
7. [Export manifest and validation checks](#manifest-validation)

<a id="environment-input"></a>

<div style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 1</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Environment and input</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Load the validated panel, set export folders and enforce the article input contract.</p>
</div>

In [1]:
from __future__ import annotations

import importlib.metadata as importlib_metadata
import importlib.util
import json
import math
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

REQUIRED_MODULES = {"pandas":"pandas", "numpy":"numpy", "matplotlib":"matplotlib", "pyarrow":"pyarrow"}
missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter, PercentFormatter
from PIL import Image

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 140)

COLORS = {
    "ink":"#102A43", "navy_2":"#243B53", "copper":"#D4A44C", "muted":"#66788A",
    "paper_2":"#EDE9DF", "border":"#D8D5CC", "white":"#FFFFFF", "paper":"#F7F4EC",
    "info_ink":"#1C4F8A", "ok":"#1F8A5B", "caution_ink":"#8A6D1F", "limit_ink":"#9C3B3B",
}
CATEGORICAL = ["#102A43", "#2F6B8A", "#3E7A78", "#7A6680", "#7B6E4B", "#66788A"]
FIGURE_PROFILES = {"inline":{"figsize":(8,5),"dpi":200}, "wide":{"figsize":(12,7.2),"dpi":200}, "full":{"figsize":(14,8),"dpi":200}}
SANS = ["IBM Plex Sans", "DejaVu Sans", "Arial", "sans-serif"]
MONO = ["IBM Plex Mono", "DejaVu Sans Mono", "monospace"]
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
SANS_USED = "IBM Plex Sans" if "IBM Plex Sans" in installed_fonts else "DejaVu Sans"
MONO_USED = "IBM Plex Mono" if "IBM Plex Mono" in installed_fonts else "DejaVu Sans Mono"
IBM_PLEX_AVAILABLE = SANS_USED == "IBM Plex Sans" and MONO_USED == "IBM Plex Mono"

PALETTE = {
    "navy":COLORS["ink"], "blue":COLORS["ink"], "orange":COLORS["copper"],
    "green":COLORS["ok"], "red":COLORS["limit_ink"], "purple":CATEGORICAL[3],
    "brown":CATEGORICAL[4], "gray":COLORS["muted"], "paper":COLORS["paper"],
}
plt.rcParams.update({
    "font.family":SANS, "font.sans-serif":SANS, "font.monospace":MONO,
    "figure.facecolor":COLORS["white"], "axes.facecolor":COLORS["white"],
    "savefig.facecolor":COLORS["white"], "savefig.transparent":False,
    "axes.edgecolor":COLORS["border"], "axes.labelcolor":COLORS["navy_2"],
    "axes.titlecolor":COLORS["ink"], "text.color":COLORS["navy_2"],
    "xtick.color":COLORS["muted"], "ytick.color":COLORS["muted"],
    "axes.spines.top":False, "axes.spines.right":False,
    "axes.grid":True, "grid.color":COLORS["paper_2"], "grid.linewidth":0.8, "grid.alpha":1.0,
    "axes.axisbelow":True, "axes.titlesize":15, "axes.labelsize":11.5,
    "xtick.labelsize":10, "ytick.labelsize":10, "legend.fontsize":9.5, "figure.dpi":120,
})

def fmt_int(value):
    if value is None or pd.isna(value): return "NA"
    return f"{float(value):,.0f}"

def fmt_num(value, decimals=2):
    if value is None or pd.isna(value): return "NA"
    return f"{float(value):,.{decimals}f}"

def fmt_pct(value, decimals=1):
    if value is None or pd.isna(value): return "NA"
    return f"{float(value)*100:.{decimals}f}%"

def display_note(title, text, kind="info"):
    colors = {
        "info":(COLORS["info_ink"],"#DCEEFF",COLORS["ink"]),
        "success":(COLORS["ok"],"#DFF3E4",COLORS["ink"]),
        "warning":(COLORS["caution_ink"],"#FFF3CD",COLORS["ink"]),
        "error":(COLORS["limit_ink"],"#FDE2E2",COLORS["ink"]),
    }
    border, background, foreground = colors[kind]
    display(HTML(f"<div style='box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;overflow-wrap:anywhere;padding:15px 18px;border-left:5px solid {border};border-radius:6px;background:{background};color:{foreground};line-height:1.55;margin:14px 0;'><strong>{title}</strong><br>{text}</div>"))

def save_table(df, filename):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    return path

FIGURE_EXPORT_CONTEXT = None
publication_figure_paths = {}

def save_fig(fig, filename, directory=None, tight_layout=True):
    context = FIGURE_EXPORT_CONTEXT
    if context is None:
        raise RuntimeError("FIGURE_EXPORT_CONTEXT must be defined before exporting a figure.")
    stem = context["stem"]
    png_path = FIGURE_DIR / f"{stem}.png"
    svg_path = FIGURE_DIR / f"{stem}.svg"
    csv_path = FIGURE_DIR / f"{stem}_data.csv"
    metadata_path = FIGURE_DIR / f"{stem}_metadata.json"
    if tight_layout:
        fig.tight_layout()
    context["data"].to_csv(csv_path, index=False)
    fig.savefig(png_path, dpi=200, bbox_inches="tight", pad_inches=.16, facecolor=COLORS["white"], transparent=False)
    fig.savefig(svg_path, bbox_inches="tight", pad_inches=.16, facecolor=COLORS["white"], transparent=False)
    with Image.open(png_path) as image:
        width_px, height_px = image.size
        preview_height = round(image.height * 360 / image.width)
    assert width_px > 0 and height_px > 0 and preview_height > 0
    metadata = {
        "figure_id":context["figure_id"], "article":ARTICLE_SLUG, "notebook":NOTEBOOK_NAME,
        "metric":context["metric"], "analytical_universe":context["universe"], "period":"1989–2025",
        "source":"UCDP Organized Violence v26.1; ConflictLens calculations.", "export_profile":context.get("profile","wide"),
        "width_px":width_px, "height_px":height_px, "dpi":200,
        "generated_at":datetime.now(timezone.utc).isoformat(), "notes":context.get("notes",""),
        "font_sans":SANS_USED, "font_mono":MONO_USED,
    }
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    publication_figure_paths[f"{stem}.png"] = png_path
    display(HTML(f"<div style='font-size:13px;color:#66788A;margin:8px 0 14px;'>Exported PNG, SVG, CSV and metadata: <code>{stem}</code></div>"))
    plt.show()
    return png_path

save_figure = save_fig

PROJECT_ROOT = Path.cwd()
if not any((PROJECT_ROOT / name).exists() for name in ["outputs", "02_analysis_unit_year_panel_analysis_ready.parquet"]):
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "outputs").exists() or (candidate / "02_analysis_unit_year_panel_analysis_ready.parquet").exists():
            PROJECT_ROOT = candidate
            break
OUTPUT_ROOT = PROJECT_ROOT / "03bis_seven_years_article_outputs"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = PROJECT_ROOT / "figures" / "seven-years-out-of-thirty-seven"
for folder in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]: folder.mkdir(parents=True, exist_ok=True)
ARTICLE_FIGURE_DIR = FIGURE_DIR
ARTICLE_SLUG = "seven-years-out-of-thirty-seven"
NOTEBOOK_NAME = "01_seven_years_out_of_thirty_seven.ipynb"
package_versions = {module:importlib_metadata.version(package) for module,package in REQUIRED_MODULES.items()}
display_note("Execution environment", f"Python <code>{sys.version.split()[0]}</code> · Matplotlib <code>{matplotlib.__version__}</code><br>Fonts: <code>{SANS_USED}</code> / <code>{MONO_USED}</code> · IBM Plex available: <code>{IBM_PLEX_AVAILABLE}</code><br>Figure directory: <code>{FIGURE_DIR}</code>")


In [2]:
PANEL_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "02_analysis_unit_year_panel_analysis_ready.parquet",
    PROJECT_ROOT / "02_analysis_unit_year_panel_analysis_ready.parquet",
    PROJECT_ROOT / "outputs" / "02_country_year_panel_analysis_ready.parquet",
    PROJECT_ROOT / "02_country_year_panel_analysis_ready.parquet",
]
STAGING_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "02_country_year_panel_staging.parquet",
    PROJECT_ROOT / "02_country_year_panel_staging.parquet",
]

PANEL_PATH = next((path for path in PANEL_CANDIDATES if path.exists()), None)
STAGING_PATH = next((path for path in STAGING_CANDIDATES if path.exists()), None)
if PANEL_PATH is None:
    raise FileNotFoundError(
        "This notebook requires the analysis-ready panel exported by Notebook 02.\n"
        "Expected one of:\n" + "\n".join(f"- {path}" for path in PANEL_CANDIDATES)
    )
if STAGING_PATH is None:
    raise FileNotFoundError(
        "This notebook requires the staging panel exported by Notebook 02 for low/best/high sensitivity.\n"
        "Expected one of:\n" + "\n".join(f"- {path}" for path in STAGING_CANDIDATES)
    )

panel_raw = pd.read_parquet(PANEL_PATH)
staging_raw = pd.read_parquet(STAGING_PATH)

REQUIRED_COLUMNS = [
    "analysis_unit_id", "analysis_unit_label", "analysis_unit_type", "country_iso3",
    "country_name", "year", "analysis_conflict_universe", "unit_exists_in_year",
    "ucdp_ov_total_deaths_best_zf", "ucdp_ov_civilian_deaths_best_zf",
    "ucdp_ov_any_organized_violence_zf", "ucdp_ged_events_zf",
    "ucdp_ged_state_based_events_zf", "ucdp_ged_non_state_events_zf",
    "ucdp_ged_one_sided_events_zf", "ucdp_ged_best_deaths_zf",
    "ucdp_ged_civilian_deaths_zf",
]
STAGING_REQUIRED_COLUMNS = [
    "analysis_unit_id", "year", "ucdp_ged_low_deaths",
    "ucdp_ged_best_deaths", "ucdp_ged_high_deaths",
]
missing_cols = [col for col in REQUIRED_COLUMNS if col not in panel_raw.columns]
missing_staging_cols = [col for col in STAGING_REQUIRED_COLUMNS if col not in staging_raw.columns]
if missing_cols:
    raise AssertionError(f"Missing required analysis-ready columns: {missing_cols}")
if missing_staging_cols:
    raise AssertionError(f"Missing required staging columns: {missing_staging_cols}")
if staging_raw.duplicated(["analysis_unit_id", "year"]).any():
    raise AssertionError("The staging panel must be unique at analysis_unit_id + year.")

panel = panel_raw.copy()
panel["year"] = panel["year"].astype(int)

numeric_cols = [
    "ucdp_ov_total_deaths_best_zf", "ucdp_ov_civilian_deaths_best_zf",
    "ucdp_ov_any_organized_violence_zf", "ucdp_ged_events_zf",
    "ucdp_ged_state_based_events_zf", "ucdp_ged_non_state_events_zf",
    "ucdp_ged_one_sided_events_zf", "ucdp_ged_best_deaths_zf",
    "ucdp_ged_civilian_deaths_zf",
]
for col in numeric_cols:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

analysis_conflict = panel[
    (panel["analysis_conflict_universe"] == True)
    & (panel["unit_exists_in_year"] == True)
].copy()

uncertainty_panel = analysis_conflict[[
    "analysis_unit_id", "analysis_unit_label", "year", "ucdp_ov_total_deaths_best_zf"
]].merge(
    staging_raw[STAGING_REQUIRED_COLUMNS],
    on=["analysis_unit_id", "year"],
    how="left",
    validate="one_to_one",
)
for col in ["ucdp_ged_low_deaths", "ucdp_ged_best_deaths", "ucdp_ged_high_deaths"]:
    uncertainty_panel[col] = pd.to_numeric(uncertainty_panel[col], errors="coerce").fillna(0.0)

best_reconciliation_gap = float(
    (
        uncertainty_panel["ucdp_ov_total_deaths_best_zf"].fillna(0.0)
        - uncertainty_panel["ucdp_ged_best_deaths"]
    ).abs().sum()
)
if not np.isclose(best_reconciliation_gap, 0.0):
    raise AssertionError(
        "The OV best-estimate metric does not reconcile with the underlying GED best estimates. "
        f"Absolute gap: {best_reconciliation_gap:,.3f}"
    )

input_summary = pd.DataFrame([
    {"metric": "analysis-ready panel rows", "value": len(panel_raw)},
    {"metric": "analysis-ready panel columns", "value": panel_raw.shape[1]},
    {"metric": "staging panel rows", "value": len(staging_raw)},
    {"metric": "analysis units", "value": panel_raw["analysis_unit_id"].nunique()},
    {"metric": "conflict-universe rows after existence guard", "value": len(analysis_conflict)},
    {"metric": "conflict-universe units after existence guard", "value": analysis_conflict["analysis_unit_id"].nunique()},
    {"metric": "year min", "value": analysis_conflict["year"].min()},
    {"metric": "year max", "value": analysis_conflict["year"].max()},
    {"metric": "OV best / GED best absolute reconciliation gap", "value": best_reconciliation_gap},
])
save_table(input_summary, "input_summary.csv")
input_summary

,metric,value
0,analysis-ready panel rows,9287.0
1,analysis-ready panel columns,66.0
2,staging panel rows,9538.0
3,analysis units,251.0
4,conflict-universe rows after existence guard,9068.0
5,conflict-universe units after existence guard,249.0
6,year min,1989.0
7,year max,2025.0
8,OV best / GED best absolute reconciliation gap,0.0


### Observed result — input and uncertainty contracts validated

The notebook loads the analysis-ready panel and the staging panel exported by Notebook 02. It applies the same existence guard as Notebook 03, so pre-existence rows are not silently interpreted as peaceful observations.

The selected OV best-estimate metric reconciles exactly with the underlying GED best estimates. The staging panel additionally supplies low and high estimates, allowing the seven-year threshold to be presented as an estimate-dependent result rather than a fixed property of the historical record.

<a id="annual-curve"></a>

<div style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 2</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Annual curve, recent clustering and estimate sensitivity</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Recalculate the annual series, identify the 2021–2025 cluster and test how the 50% threshold changes across UCDP low, best and high estimates.</p>
</div>

In [3]:
annual = (
    analysis_conflict
    .groupby("year", as_index=False)
    .agg(
        ucdp_ov_total_deaths=("ucdp_ov_total_deaths_best_zf", "sum"),
        ucdp_ov_civilian_deaths=("ucdp_ov_civilian_deaths_best_zf", "sum"),
        units_with_any_ov=("ucdp_ov_any_organized_violence_zf", lambda s: int((s.fillna(0) > 0).sum())),
        units_with_ov_fatalities=("ucdp_ov_total_deaths_best_zf", lambda s: int((s.fillna(0) > 0).sum())),
    )
    .sort_values("year")
    .reset_index(drop=True)
)

expected_years = list(range(1989, 2026))
missing_years = sorted(set(expected_years) - set(annual["year"].tolist()))
if missing_years:
    raise AssertionError(f"Annual series is incomplete. Missing years: {missing_years}")

annual_uncertainty = (
    uncertainty_panel
    .groupby("year", as_index=False)
    .agg(
        low_estimate=("ucdp_ged_low_deaths", "sum"),
        best_estimate=("ucdp_ged_best_deaths", "sum"),
        high_estimate=("ucdp_ged_high_deaths", "sum"),
    )
    .sort_values("year")
    .reset_index(drop=True)
)
if not np.allclose(annual_uncertainty["best_estimate"], annual["ucdp_ov_total_deaths"]):
    raise AssertionError("Annual OV best estimates must reconcile with annual GED best estimates.")

annual_total = annual["ucdp_ov_total_deaths"].sum()
annual_ranked = annual.sort_values("ucdp_ov_total_deaths", ascending=False).reset_index(drop=True)
annual_ranked["rank_desc_total"] = annual_ranked.index + 1
annual_ranked["cumulative_deaths"] = annual_ranked["ucdp_ov_total_deaths"].cumsum()
annual_ranked["cumulative_share"] = annual_ranked["cumulative_deaths"] / annual_total
annual_years_to_50 = int(annual_ranked.loc[annual_ranked["cumulative_share"] >= 0.50, "rank_desc_total"].iloc[0])
annual_years_50 = annual_ranked.head(annual_years_to_50).copy()

def annual_estimate_profile(column, label):
    ranked = annual_uncertainty[["year", column]].sort_values(column, ascending=False).reset_index(drop=True)
    ranked["cumulative_share"] = ranked[column].cumsum() / ranked[column].sum()
    years_to_50 = int(np.searchsorted(ranked["cumulative_share"].to_numpy(), 0.50, side="left") + 1)
    return {
        "estimate": label,
        "period_total": float(ranked[column].sum()),
        "years_to_cross_50pct": years_to_50,
        "share_at_crossing": float(ranked.loc[years_to_50 - 1, "cumulative_share"]),
        "years_in_crossing_set": ranked.head(years_to_50)["year"].astype(int).tolist(),
    }

annual_estimate_sensitivity = pd.DataFrame([
    annual_estimate_profile("low_estimate", "low"),
    annual_estimate_profile("best_estimate", "best"),
    annual_estimate_profile("high_estimate", "high"),
])

recent_years = list(range(2021, 2026))
recent_2021_2025_total = float(annual.loc[annual["year"].isin(recent_years), "ucdp_ov_total_deaths"].sum())
recent_2021_2025_share = recent_2021_2025_total / annual_total
recent_years_in_best_top7 = sorted(set(recent_years) & set(annual_years_50["year"].astype(int)))

def rank_correlation(left, right):
    return float(pd.Series(left).rank().corr(pd.Series(right).rank()))

trend_diagnostics = pd.DataFrame([
    {
        "window": "1989–2025",
        "years": len(annual),
        "spearman_year_vs_deaths": rank_correlation(annual["year"], annual["ucdp_ov_total_deaths"]),
    },
    {
        "window": "1989–2025 excluding 1994",
        "years": len(annual) - 1,
        "spearman_year_vs_deaths": rank_correlation(
            annual.loc[annual["year"] != 1994, "year"],
            annual.loc[annual["year"] != 1994, "ucdp_ov_total_deaths"],
        ),
    },
    {
        "window": "1989–2020",
        "years": int((annual["year"] <= 2020).sum()),
        "spearman_year_vs_deaths": rank_correlation(
            annual.loc[annual["year"] <= 2020, "year"],
            annual.loc[annual["year"] <= 2020, "ucdp_ov_total_deaths"],
        ),
    },
    {
        "window": "2000–2025",
        "years": int((annual["year"] >= 2000).sum()),
        "spearman_year_vs_deaths": rank_correlation(
            annual.loc[annual["year"] >= 2000, "year"],
            annual.loc[annual["year"] >= 2000, "ucdp_ov_total_deaths"],
        ),
    },
])

save_table(annual, "annual_ucdp_ov_curve_1989_2025.csv")
save_table(annual_uncertainty, "annual_ucdp_estimate_sensitivity.csv")
save_table(annual_estimate_sensitivity, "annual_years_to_50pct_by_estimate.csv")
save_table(trend_diagnostics, "annual_rank_trend_diagnostics.csv")
save_table(annual_ranked, "annual_concentration_ranked.csv")
save_table(annual_years_50, "annual_years_needed_to_reach_50pct_best_estimate.csv")

display(annual_ranked.head(12))
annual_estimate_sensitivity

,year,ucdp_ov_total_deaths,ucdp_ov_civilian_deaths,units_with_any_ov,units_with_ov_fatalities,rank_desc_total,cumulative_deaths,cumulative_share
0,1994,824177.0,783026.0,61,58,1,824177.0,0.193565
1,2022,317936.0,34989.0,67,67,2,1142113.0,0.268234
2,2025,245464.0,91566.0,60,58,3,1387577.0,0.325884
3,2021,235007.0,17058.0,65,64,4,1622584.0,0.381077
4,2024,188209.0,34895.0,61,61,5,1810793.0,0.425279
5,2023,170432.0,36026.0,65,64,6,1981225.0,0.465307
6,2014,151743.0,35411.0,51,50,7,2132968.0,0.500945
7,2015,132027.0,29895.0,57,54,8,2264995.0,0.531952
8,2016,116723.0,26588.0,59,59,9,2381718.0,0.559366
9,2013,115556.0,33132.0,49,49,10,2497274.0,0.586505


,estimate,period_total,years_to_cross_50pct,share_at_crossing,years_in_crossing_set
0,low,3509097.0,10,0.530597,"[1994, 2022, 2021, 2025, 2023, 2024, 2014, 2015, 2016, 2013]"
1,best,4257891.0,7,0.500945,"[1994, 2022, 2025, 2021, 2024, 2023, 2014]"
2,high,6222204.0,6,0.519654,"[1994, 2022, 2025, 2021, 2023, 2024]"


In [4]:
fig, ax = plt.subplots(figsize=FIGURE_PROFILES["wide"]["figsize"])
ax.axvspan(2020.5, 2025.5, color=PALETTE["orange"], alpha=0.12, zorder=0)
ax.plot(annual["year"], annual["ucdp_ov_total_deaths"], linewidth=2.2, marker="o", markersize=3.5, color=PALETTE["blue"])
ax.set_title("Annual UCDP organized-violence deaths — best estimate")
ax.set_xlabel("Year")
ax.set_ylabel("Best-estimate deaths")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: fmt_int(x)))
ax.set_xlim(1988.5, 2025.5)
ax.text(2023, 0.94, "2021–2025 cluster", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=9.5, color=COLORS["caution_ink"], fontweight="bold")

for year in [1994, 2014, 2022, 2025]:
    row = annual.loc[annual["year"] == year].iloc[0]
    ax.scatter([year], [row["ucdp_ov_total_deaths"]], s=70, color=PALETTE["orange"], zorder=5)
    ax.annotate(
        f"{year}\n{fmt_int(row['ucdp_ov_total_deaths'])}",
        xy=(year, row["ucdp_ov_total_deaths"]),
        xytext=(0, 12 if year != 1994 else -48),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        arrowprops={"arrowstyle": "-", "alpha": 0.5},
    )

FIGURE_EXPORT_CONTEXT = {
    "stem":"figure_01_annual_recorded_deaths", "figure_id":"figure_01", "data":annual,
    "metric":"Annual sum of UCDP organized-violence deaths (best estimate)",
    "universe":"Existence-bound conflict universe, annual global aggregation",
    "notes":"Best-estimate series. The 2021–2025 interval is highlighted because five consecutive recent years belong to the seven-year headline set.",
}
fig1_path = save_fig(fig, "figure_01_annual_recorded_deaths.png")

findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font famil

In [5]:
fig, ax1 = plt.subplots(figsize=FIGURE_PROFILES["wide"]["figsize"])
bar_positions = np.arange(len(annual_ranked))
bar_colors = [PALETTE["orange"] if int(year) in recent_years else PALETTE["blue"] for year in annual_ranked["year"]]
ax1.bar(bar_positions, annual_ranked["ucdp_ov_total_deaths"], color=bar_colors, alpha=0.84)
ax1.set_title("Under UCDP’s best estimates, seven years exceed 50% of the total")
ax1.set_xlabel("Years sorted by descending best-estimate deaths")
ax1.set_ylabel("Best-estimate deaths")
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: fmt_int(x)))
ax1.set_xticks(bar_positions)
ax1.set_xticklabels(annual_ranked["year"].astype(str), rotation=75, ha="right")

ax2 = ax1.twinx()
ax2.plot(bar_positions, annual_ranked["cumulative_share"], color=PALETTE["orange"], marker="o", linewidth=2)
ax2.axhline(0.50, color=COLORS["caution_ink"], linestyle="--", linewidth=1.4, alpha=0.85)
ax2.set_ylabel("Cumulative share of best-estimate total")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

threshold_idx = annual_years_to_50 - 1
ax1.axvline(threshold_idx, color=COLORS["caution_ink"], linestyle=":", linewidth=1.5)
ax2.annotate(
    f"50% crossed at rank {annual_years_to_50}\n({int(annual_ranked.loc[threshold_idx, 'year'])})",
    xy=(threshold_idx, annual_ranked.loc[threshold_idx, "cumulative_share"]),
    xytext=(threshold_idx + 1.2, 0.58),
    arrowprops={"arrowstyle": "->", "color": PALETTE["red"]},
    fontsize=10,
    color=COLORS["caution_ink"],
)
ax1.text(
    0.985, 0.94,
    "Estimate sensitivity\nLow: 10 years · Best: 7 · High: 6",
    transform=ax1.transAxes, ha="right", va="top", fontsize=9.5,
    color=COLORS["muted"], bbox={"facecolor": COLORS["white"], "edgecolor": COLORS["border"], "boxstyle": "round,pad=.45"},
)

FIGURE_EXPORT_CONTEXT = {
    "stem":"figure_02_seven_year_concentration", "figure_id":"figure_02", "data":annual_ranked,
    "metric":"Ranked annual best-estimate deaths and cumulative share of the 1989–2025 total",
    "universe":"37 annual global totals, 1989–2025",
    "notes":"The seven-year threshold applies to best estimates. Low and high estimates require 10 and 6 years respectively. Recent 2021–2025 years are highlighted.",
}
fig2_path = save_fig(fig, "figure_02_seven_year_concentration.png")

findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font famil

### Observed result — the headline is valid for the best estimate, not invariant to uncertainty

Across 1989–2025, the best-estimate series sums to **4,257,891 UCDP organized-violence deaths**. Sorting the 37 annual totals shows that **7 years** exceed 50% of that best-estimate total: **1994, 2022, 2025, 2021, 2024, 2023 and 2014**. Together they sum to **2,132,968 deaths**, or **50.1%**.

The exact cutoff changes across UCDP fatality estimates:

- **Low estimate:** 10 years are needed to cross 50%.
- **Best estimate:** 7 years are needed.
- **High estimate:** 6 years are needed.

The headline must therefore remain explicitly tied to the best estimate. The broader concentration finding survives, but “seven” is not uncertainty-invariant.

The seven years are also not scattered evenly across the full window. **Five are the consecutive years 2021–2025**, which together account for **27.2%** of the best-estimate period total. This recent cluster coexists with the exceptional 1994 shock. A single linear trend cannot summarise both structures, but the presence of extreme years does not justify saying that the series has no temporal pattern.

<a id="country-year-concentration"></a>

<div style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 3</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Country-year concentration</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Validate the unit-year concentration statistics used as supporting evidence.</p>
</div>

In [6]:
positive_unit_years = (
    analysis_conflict[analysis_conflict["ucdp_ov_total_deaths_best_zf"].fillna(0) > 0]
    .sort_values("ucdp_ov_total_deaths_best_zf", ascending=False)
    .reset_index(drop=True)
    .copy()
)
positive_unit_years["rank_desc_total"] = positive_unit_years.index + 1
positive_unit_years["cumulative_deaths"] = positive_unit_years["ucdp_ov_total_deaths_best_zf"].cumsum()
positive_unit_years["cumulative_share"] = positive_unit_years["cumulative_deaths"] / positive_unit_years["ucdp_ov_total_deaths_best_zf"].sum()
unit_years_to_50 = int(positive_unit_years.loc[positive_unit_years["cumulative_share"] >= 0.50, "rank_desc_total"].iloc[0])
unit_years_50 = positive_unit_years.head(unit_years_to_50).copy()

concentration_rows = []
for pct in [0.01, 0.05, 0.10]:
    n = int(math.ceil(len(positive_unit_years) * pct))
    concentration_rows.append({
        "metric": "ucdp_ov_total_deaths_best_zf",
        "universe": "positive best-estimate unit-years",
        "positive_unit_years": len(positive_unit_years),
        "top_bucket": f"top_{int(pct * 100)}pct",
        "rows_in_bucket": n,
        "share_of_total": positive_unit_years.head(n)["ucdp_ov_total_deaths_best_zf"].sum() / positive_unit_years["ucdp_ov_total_deaths_best_zf"].sum(),
    })

positive_civilian_unit_years = (
    analysis_conflict[analysis_conflict["ucdp_ov_civilian_deaths_best_zf"].fillna(0) > 0]
    .sort_values("ucdp_ov_civilian_deaths_best_zf", ascending=False)
    .reset_index(drop=True)
    .copy()
)
civ_top_1pct_n = int(math.ceil(len(positive_civilian_unit_years) * 0.01))
concentration_rows.append({
    "metric": "ucdp_ov_civilian_deaths_best_zf",
    "universe": "positive civilian-death unit-years",
    "positive_unit_years": len(positive_civilian_unit_years),
    "top_bucket": "top_1pct",
    "rows_in_bucket": civ_top_1pct_n,
    "share_of_total": positive_civilian_unit_years.head(civ_top_1pct_n)["ucdp_ov_civilian_deaths_best_zf"].sum() / positive_civilian_unit_years["ucdp_ov_civilian_deaths_best_zf"].sum(),
})

def unit_year_estimate_profile(column, label):
    ranked = uncertainty_panel.loc[uncertainty_panel[column] > 0, column].sort_values(ascending=False).reset_index(drop=True)
    cumulative_share = ranked.cumsum() / ranked.sum()
    rows_to_50 = int(np.searchsorted(cumulative_share.to_numpy(), 0.50, side="left") + 1)
    top_1pct_rows = int(math.ceil(len(ranked) * 0.01))
    return {
        "estimate": label,
        "positive_unit_years": len(ranked),
        "unit_years_to_cross_50pct": rows_to_50,
        "share_positive_rows_to_cross_50pct": rows_to_50 / len(ranked),
        "top_1pct_rows": top_1pct_rows,
        "top_1pct_share": float(ranked.head(top_1pct_rows).sum() / ranked.sum()),
    }

unit_year_estimate_sensitivity = pd.DataFrame([
    unit_year_estimate_profile("ucdp_ged_low_deaths", "low"),
    unit_year_estimate_profile("ucdp_ged_best_deaths", "best"),
    unit_year_estimate_profile("ucdp_ged_high_deaths", "high"),
])

concentration_by_percentile = pd.DataFrame(concentration_rows)
save_table(concentration_by_percentile, "unit_year_concentration_by_percentile.csv")
save_table(unit_year_estimate_sensitivity, "unit_year_concentration_by_estimate.csv")
save_table(unit_years_50[[
    "rank_desc_total", "analysis_unit_id", "analysis_unit_label", "year",
    "ucdp_ov_total_deaths_best_zf", "ucdp_ov_civilian_deaths_best_zf", "cumulative_share"
]], "top_positive_unit_years_needed_to_reach_50pct_best_estimate.csv")

display(concentration_by_percentile)
display(unit_year_estimate_sensitivity)
unit_years_50[["rank_desc_total", "analysis_unit_label", "year", "ucdp_ov_total_deaths_best_zf", "cumulative_share"]].tail(8)

,metric,universe,positive_unit_years,top_bucket,rows_in_bucket,share_of_total
0,ucdp_ov_total_deaths_best_zf,positive best-estimate unit-years,2067,top_1pct,21,0.497965
1,ucdp_ov_total_deaths_best_zf,positive best-estimate unit-years,2067,top_5pct,104,0.718574
2,ucdp_ov_total_deaths_best_zf,positive best-estimate unit-years,2067,top_10pct,207,0.818707
3,ucdp_ov_civilian_deaths_best_zf,positive civilian-death unit-years,1730,top_1pct,18,0.695057


,estimate,positive_unit_years,unit_years_to_cross_50pct,share_positive_rows_to_cross_50pct,top_1pct_rows,top_1pct_share
0,low,2056,30,0.014591,21,0.445290
1,best,2067,22,0.010643,21,0.497965
2,high,2132,17,0.007974,22,0.530872


,rank_desc_total,analysis_unit_label,year,ucdp_ov_total_deaths_best_zf,cumulative_share
14,15,Ethiopia,2000,48666.0,0.451470
15,16,Syrian Arab Republic,2017,38532.0,0.460520
16,17,Afghanistan,2021,36386.0,0.469066
17,18,"Congo, The Democratic Republic of the",1996,34296.0,0.477120
18,19,Ethiopia,1999,30786.0,0.484351
19,20,Afghanistan,2019,30435.0,0.491498
20,21,Israel,2023,27535.0,0.497965
21,22,Afghanistan,2018,26895.0,0.504282


### Observed result — the denominator and estimate must be named

The existence-bound conflict universe contains **9,068 unit-years**, of which **2,067** have at least one best-estimate OV death. Within that **positive** distribution, **22 unit-years** — 1.06% of positive rows, or 0.24% of all existence-bound rows — cross 50% of best-estimate deaths.

Among positive best-estimate unit-years, the top **1%**, **5%** and **10%** account for **49.8%**, **71.9%** and **81.9%** of deaths respectively. Among **1,730 positive civilian-death unit-years**, the top 1% account for **69.5%** of civilian deaths.

The exact unit-year threshold is estimate-dependent: **30 low-estimate**, **22 best-estimate** and **17 high-estimate** unit-years cross half of their corresponding totals. Concentration is robust as a broad finding; its exact magnitude is not fixed.

<a id="peak-contributors"></a>

<div style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 4</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Peak-year contributor structures and Israel 2025</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Decompose selected peak years while separating measured contribution and civilian share from historical, legal or perpetrator classification.</p>
</div>

### Reading note — contributor does not mean perpetrator

A row such as `Sudan — 2025`, `Ukraine — 2025` or `Israel — 2025` means that deaths are coded under that analytical unit and year in the panel. It does **not** identify perpetrator responsibility, victim nationality, legal classification, event location logic, or target identity. Those questions require actor- and event-level historical analysis.

In [7]:
peak_years = [1994, 2014, 2022, 2025]
peak_tables = []

for year in peak_years:
    sub = analysis_conflict[analysis_conflict["year"] == year].copy()
    annual_total_y = sub["ucdp_ov_total_deaths_best_zf"].sum()
    annual_civilian_y = sub["ucdp_ov_civilian_deaths_best_zf"].sum()
    sub["annual_ov_total_deaths"] = annual_total_y
    sub["annual_ov_civilian_deaths"] = annual_civilian_y
    sub["share_of_annual_total"] = sub["ucdp_ov_total_deaths_best_zf"].fillna(0) / annual_total_y
    sub["share_of_annual_civilian_deaths"] = np.where(
        annual_civilian_y > 0,
        sub["ucdp_ov_civilian_deaths_best_zf"].fillna(0) / annual_civilian_y,
        np.nan,
    )
    sub = sub.sort_values("ucdp_ov_total_deaths_best_zf", ascending=False).reset_index(drop=True)
    sub["rank_in_year"] = sub.index + 1
    peak_tables.append(sub.head(10))

peak_contributors = pd.concat(peak_tables, ignore_index=True)
save_table(peak_contributors[[
    "year", "rank_in_year", "analysis_unit_id", "analysis_unit_label", "ucdp_ov_total_deaths_best_zf",
    "share_of_annual_total", "ucdp_ov_civilian_deaths_best_zf", "share_of_annual_civilian_deaths",
    "annual_ov_total_deaths", "annual_ov_civilian_deaths"
]], "peak_year_top10_contributors.csv")

peak_contributors[[
    "year", "rank_in_year", "analysis_unit_label", "ucdp_ov_total_deaths_best_zf", "share_of_annual_total",
    "ucdp_ov_civilian_deaths_best_zf", "share_of_annual_civilian_deaths"
]].head(40)

,year,rank_in_year,analysis_unit_label,ucdp_ov_total_deaths_best_zf,share_of_annual_total,ucdp_ov_civilian_deaths_best_zf,share_of_annual_civilian_deaths
0,1994,1,Rwanda,772463.0,0.937254,771118.0,0.984792
1,1994,2,Afghanistan,9056.0,0.010988,120.0,0.000153
2,1994,3,Bosnia and Herzegovina,7312.0,0.008872,1241.0,0.001585
3,1994,4,Türkiye,4182.0,0.005074,344.0,0.000439
4,1994,5,Liberia,4167.0,0.005056,3733.0,0.004767
5,1994,6,Angola,4009.0,0.004864,304.0,0.000388
6,1994,7,Sierra Leone,2097.0,0.002544,1551.0,0.001981
7,1994,8,Ghana,2004.0,0.002432,0.0,0.000000
8,1994,9,Algeria,1943.0,0.002358,30.0,0.000038
9,1994,10,Azerbaijan,1489.0,0.001807,10.0,0.000013


In [8]:
selected_story_cases = [
    (1994, "Rwanda"),
    (2014, "Syrian Arab Republic"),
    (2014, "Iraq"),
    (2022, "Ethiopia"),
    (2022, "Ukraine"),
    (2025, "Ukraine"),
    (2025, "Sudan"),
]

story_rows = []
for year, label in selected_story_cases:
    row = peak_contributors[(peak_contributors["year"] == year) & (peak_contributors["analysis_unit_label"] == label)]
    if row.empty:
        raise AssertionError(f"Missing selected contributor: {year} / {label}")
    r = row.iloc[0].copy()
    r["story_label"] = f"{year} · {label}"
    r["coded_civilian_share_within_contributor"] = (
        r["ucdp_ov_civilian_deaths_best_zf"] / r["ucdp_ov_total_deaths_best_zf"]
        if r["ucdp_ov_total_deaths_best_zf"] > 0 else np.nan
    )
    story_rows.append(r)

peak_story_contributors = pd.DataFrame(story_rows).sort_values(["year", "share_of_annual_total"], ascending=[True, False])
save_table(peak_story_contributors[[
    "year", "analysis_unit_label", "ucdp_ov_total_deaths_best_zf", "share_of_annual_total",
    "ucdp_ov_civilian_deaths_best_zf", "coded_civilian_share_within_contributor",
    "share_of_annual_civilian_deaths",
]], "selected_peak_contributor_profiles.csv")

fig, ax = plt.subplots(figsize=FIGURE_PROFILES["wide"]["figsize"])
plot_df = peak_story_contributors.sort_values("share_of_annual_total", ascending=True).copy()
colors = [
    plt.get_cmap("Blues")(0.35 + 0.55 * float(value))
    for value in plot_df["coded_civilian_share_within_contributor"].fillna(0)
]
ax.barh(plot_df["story_label"], plot_df["share_of_annual_total"], color=colors, alpha=0.9)
ax.set_title("Selected peaks have different contributor and civilian-share profiles")
ax.set_xlabel("Contributor share of annual best-estimate deaths")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel("")

for _, row in plot_df.iterrows():
    ax.text(
        row["share_of_annual_total"] + 0.008,
        row["story_label"],
        f"{fmt_pct(row['share_of_annual_total'])} annual · {fmt_pct(row['coded_civilian_share_within_contributor'])} coded civilian",
        va="center", fontsize=8.8,
    )

FIGURE_EXPORT_CONTEXT = {
    "stem":"figure_03_peak_contributors_by_profile", "figure_id":"figure_03", "data":peak_story_contributors,
    "metric":"Selected contributor share of annual deaths and coded civilian share within contributor",
    "universe":"Selected contributors in 1994, 2014, 2022 and 2025",
    "notes":"No historical, legal or perpetrator typology is inferred. Colour encodes the contributor-level coded civilian share.",
}
fig3_path = save_fig(fig, "figure_03_peak_contributors_by_profile.png")

peak_story_contributors[[
    "year", "analysis_unit_label", "share_of_annual_total", "coded_civilian_share_within_contributor"
]]

findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font famil

,year,analysis_unit_label,share_of_annual_total,coded_civilian_share_within_contributor
0,1994,Rwanda,0.937254,0.998259
10,2014,Syrian Arab Republic,0.518699,0.229567
11,2014,Iraq,0.108321,0.205390
20,2022,Ethiopia,0.519051,0.014798
21,2022,Ukraine,0.316428,0.191553
30,2025,Ukraine,0.368058,0.029166
31,2025,Sudan,0.306253,0.876779


In [9]:
israel_2025 = peak_contributors[(peak_contributors["year"] == 2025) & (peak_contributors["analysis_unit_label"] == "Israel")].copy()
if israel_2025.empty:
    raise AssertionError("Israel 2025 row not found in the peak contributor table.")

save_table(israel_2025[[
    "year", "rank_in_year", "analysis_unit_label", "ucdp_ov_total_deaths_best_zf", "share_of_annual_total",
    "ucdp_ov_civilian_deaths_best_zf", "share_of_annual_civilian_deaths"
]], "israel_2025_raw_contributor_metrics_only.csv")

israel_2025[[
    "year", "rank_in_year", "analysis_unit_label", "ucdp_ov_total_deaths_best_zf", "share_of_annual_total",
    "ucdp_ov_civilian_deaths_best_zf", "share_of_annual_civilian_deaths"
]]

,year,rank_in_year,analysis_unit_label,ucdp_ov_total_deaths_best_zf,share_of_annual_total,ucdp_ov_civilian_deaths_best_zf,share_of_annual_civilian_deaths
32,2025,3,Israel,14563.0,0.059328,3433.0,0.037492


### Observed result — selected peaks differ in measured structure

The selected contributor checks reproduce the article’s arithmetic while avoiding unsupported historical classification:

- **1994:** Rwanda accounts for **93.7%** of annual best-estimate deaths; **99.8%** of Rwanda’s coded total is civilian.
- **2014:** Syrian Arab Republic accounts for **51.9%** of the annual total and Iraq for **10.8%**; Syria’s coded civilian share is **23.0%**.
- **2022:** Ethiopia accounts for **51.9%** and Ukraine for **31.6%**; their coded civilian shares are **1.5%** and **19.2%** respectively.
- **2025:** Ukraine accounts for **36.8%** and Sudan for **30.6%**; their coded civilian shares are **2.9%** and **87.7%** respectively. Sudan carries **72.0%** of the year’s coded civilian deaths.
- **Israel 2025:** rank **3**, **14,563** best-estimate deaths and **5.9%** of the annual total.

These values establish that similarly prominent contributors can have very different recorded profiles. They do **not** independently establish whether a case is a genocide, civil war or interstate war. Those classifications require source variables and historical evidence outside this aggregate comparison. Israel 2025 remains excluded from the GED event-composition figure as an explicit public-facing guardrail.

In [10]:
from matplotlib.patches import FancyBboxPatch

expected_2025_units = ["Ukraine", "Sudan", "Israel"]
contributors_2025 = peak_contributors[peak_contributors["year"] == 2025].copy()
expected_row_counts = (
    contributors_2025["analysis_unit_label"]
    .value_counts()
    .reindex(expected_2025_units, fill_value=0)
)
assert (expected_row_counts == 1).all(), f"Expected each 2025 analytical-unit row exactly once: {expected_row_counts.to_dict()}"

rank_by_unit_2025 = contributors_2025.set_index("analysis_unit_label")["rank_in_year"].to_dict()
assert int(rank_by_unit_2025["Ukraine"]) == 1
assert int(rank_by_unit_2025["Sudan"]) == 2
assert int(rank_by_unit_2025["Israel"]) == 3

annual_2025_row = annual.loc[annual["year"] == 2025, "ucdp_ov_total_deaths"]
assert len(annual_2025_row) == 1, "The independently aggregated annual table must contain exactly one 2025 row."
annual_total_2025 = float(annual_2025_row.iloc[0])
assert annual_total_2025 > 0, "The 2025 annual total must be non-zero."

displayed_annual_total_2025 = float(contributors_2025["annual_ov_total_deaths"].iloc[0])
assert np.isclose(displayed_annual_total_2025, annual_total_2025), (
    "The figure total does not match the notebook's independently computed 2025 annual total."
)

selected_2025 = (
    contributors_2025[contributors_2025["analysis_unit_label"].isin(expected_2025_units)]
    .set_index("analysis_unit_label")
)
display_labels = {
    "Ukraine": "Ukraine",
    "Sudan": "Sudan",
    "Israel": "Israel–2025 analytical unit",
}
segment_rows = [
    {
        "segment": display_labels[label],
        "death_total": float(selected_2025.loc[label, "ucdp_ov_total_deaths_best_zf"]),
    }
    for label in expected_2025_units
]
other_deaths_2025 = annual_total_2025 - sum(row["death_total"] for row in segment_rows)
segment_rows.append({"segment": "Other analytical units", "death_total": other_deaths_2025})

country_year_grouping_2025 = pd.DataFrame(segment_rows)
country_year_grouping_2025["share_of_annual_total"] = (
    country_year_grouping_2025["death_total"] / annual_total_2025
)

assert np.isclose(country_year_grouping_2025["death_total"].sum(), annual_total_2025)
assert np.isclose(country_year_grouping_2025["share_of_annual_total"].sum(), 1.0, atol=1e-12)
assert (country_year_grouping_2025["death_total"] >= 0).all()
assert (country_year_grouping_2025["share_of_annual_total"] >= 0).all()
save_table(country_year_grouping_2025, "fig04_country_year_grouping_2025.csv")

segment_colors = {
    "Ukraine": CATEGORICAL[0],
    "Sudan": CATEGORICAL[1],
    "Israel–2025 analytical unit": COLORS["copper"],
    "Other analytical units": COLORS["paper_2"],
}
paper = COLORS["white"]
ink = COLORS["ink"]
muted = COLORS["muted"]

fig = plt.figure(figsize=FIGURE_PROFILES["wide"]["figsize"], facecolor=COLORS["white"])
grid = fig.add_gridspec(
    1, 2, width_ratios=[1.68, 1.0], wspace=0.18,
    left=0.055, right=0.97, bottom=0.09, top=0.76,
)
ax_left = fig.add_subplot(grid[0, 0])
ax_right = fig.add_subplot(grid[0, 1])

fig.text(0.055, 0.945, "Country-year contribution is not attribution", fontsize=19, fontweight="bold", color=ink, va="top")
fig.text(
    0.055, 0.895,
    "Shares represent statistical assignments to analytical units, not perpetrator attributions.",
    fontsize=11.5, color=muted, va="top",
)

ax_left.set_facecolor(paper)
ax_left.set_xlim(0, 1)
ax_left.set_ylim(-0.18, 1.0)
ax_left.axis("off")
ax_left.text(0, 0.97, "How the 2025 total is grouped in the country-year panel", fontsize=13, fontweight="bold", color=ink, va="top")
ax_left.text(0, 0.84, f"{displayed_annual_total_2025:,.0f} recorded organized-violence deaths", fontsize=10.5, color=muted, va="top")

bar_y = 0.55
bar_height = 0.28
left_edge = 0.0
segment_centres = {}
for _, row in country_year_grouping_2025.iterrows():
    width = float(row["share_of_annual_total"])
    label = row["segment"]
    ax_left.barh(
        bar_y, width, left=left_edge, height=bar_height,
        color=segment_colors[label], edgecolor=paper, linewidth=1.5,
    )
    centre = left_edge + width / 2
    segment_centres[label] = centre
    if label != "Israel–2025 analytical unit":
        label_color = "white" if label in ["Ukraine", "Sudan"] else ink
        ax_left.text(
            centre, bar_y,
            f"{label}\n{row['share_of_annual_total']:.1%} · {row['death_total']:,.0f}",
            ha="center", va="center", fontsize=9.2, fontweight="semibold", color=label_color,
        )
    left_edge += width

israel_row = country_year_grouping_2025.loc[
    country_year_grouping_2025["segment"] == "Israel–2025 analytical unit"
].iloc[0]
ax_left.annotate(
    f"Israel–2025 analytical unit\n{israel_row['share_of_annual_total']:.1%} · {israel_row['death_total']:,.0f}",
    xy=(segment_centres["Israel–2025 analytical unit"], bar_y - bar_height / 2),
    xytext=(segment_centres["Israel–2025 analytical unit"], 0.20),
    ha="center", va="top", fontsize=9.2, fontweight="semibold", color=COLORS["caution_ink"],
    arrowprops={"arrowstyle": "-", "color": COLORS["copper"], "linewidth": 1.3},
)
ax_left.text(1.0, 0.35, "100% of the annual total", ha="right", va="top", fontsize=9, color=muted)

ax_right.set_xlim(0, 1)
ax_right.set_ylim(0, 1)
ax_right.axis("off")
ax_right.set_facecolor(paper)

def add_interpretation_block(ax, y, heading, items, accent, background):
    box = FancyBboxPatch(
        (0.02, y), 0.96, 0.34,
        boxstyle="round,pad=0.018,rounding_size=0.02",
        linewidth=0.8, edgecolor=COLORS["border"], facecolor=background,
    )
    ax.add_patch(box)
    ax.add_patch(FancyBboxPatch((0.02, y), 0.018, 0.34, boxstyle="round,pad=0,rounding_size=0.008", linewidth=0, facecolor=accent))
    ax.text(0.075, y + 0.285, heading, fontsize=9.7, fontweight="bold", color=ink, va="center")
    for idx, item in enumerate(items):
        ax.text(0.08, y + 0.225 - idx * 0.052, f"•  {item}", fontsize=9.3, color=muted, va="center")

add_interpretation_block(
    ax_right, 0.59, "WHAT THIS GROUPING TELLS US",
    ["Analytical unit", "Year", "Recorded death total", "Share of the annual total"],
    COLORS["info_ink"], "#F2F6FA",
)
add_interpretation_block(
    ax_right, 0.19, "WHAT IT DOES NOT TELL US",
    ["Who killed whom", "Who was targeted", "Victim nationality", "Legal or historical classification"],
    COLORS["muted"], "#F4F3F0",
)
ax_right.text(
    0.02, 0.075,
    "A country-year is an aggregation key,\nnot a finding about responsibility.",
    fontsize=11.2, fontweight="bold", color=ink, va="top",
)

FIGURE_EXPORT_CONTEXT = {'stem':'figure_04_country_year_contribution_2025', 'figure_id':'figure_04', 'data':country_year_grouping_2025, 'metric':'Share of 2025 deaths assigned to selected analytical units', 'universe':'Existence-bound conflict universe in 2025', 'notes':'Original two-panel explanatory composition, totals, segment values, Israel annotation, and interpretation blocks preserved.'}
fig4_path = save_fig(
    fig,
    "fig04_country_year_contribution_not_attribution_2025.png",
    directory=ARTICLE_FIGURE_DIR,
    tight_layout=False,
)

fig4_validation = pd.DataFrame([
    {"check": "2025 annual total is non-zero", "status": "passed", "value": f"{annual_total_2025:,.0f}"},
    {"check": "four death segments sum to annual total", "status": "passed", "value": f"{country_year_grouping_2025['death_total'].sum():,.0f}"},
    {"check": "four shares sum to 100%", "status": "passed", "value": f"{country_year_grouping_2025['share_of_annual_total'].sum():.1%}"},
    {"check": "2025 ranks", "status": "passed", "value": "Ukraine 1 · Sudan 2 · Israel 3"},
    {"check": "expected rows found exactly once", "status": "passed", "value": expected_row_counts.to_dict()},
    {"check": "all segment values are non-negative", "status": "passed", "value": True},
    {"check": "display total matches independent annual total", "status": "passed", "value": np.isclose(displayed_annual_total_2025, annual_total_2025)},
])
display(fig4_validation)
display_note(
    "Article Figure 4 validation complete.",
    f"Four computed segments reconcile to {annual_total_2025:,.0f} deaths and 100.0%; Ukraine, Sudan and Israel rank first, second and third.",
    kind="success",
)
country_year_grouping_2025

findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Failed to find font weight semibold, now using 700.
findfont: Failed to find font weight semibold, now using 700.
findfont: Failed to find font weight semibold, now using 700.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Ple

,check,status,value
0,2025 annual total is non-zero,passed,"245,464"
1,four death segments sum to annual total,passed,"245,464"
2,four shares sum to 100%,passed,100.0%
3,2025 ranks,passed,Ukraine 1 · Sudan 2 · Israel 3
4,expected rows found exactly once,passed,"{'Ukraine': 1, 'Sudan': 1, 'Israel': 1}"
5,all segment values are non-negative,passed,True
6,display total matches independent annual total,passed,True


,segment,death_total,share_of_annual_total
0,Ukraine,90345.0,0.368058
1,Sudan,75174.0,0.306253
2,Israel–2025 analytical unit,14563.0,0.059328
3,Other analytical units,65382.0,0.266361


### Observed result — Article Figure 4 separates contribution from attribution

The four-row display table is calculated from the validated 2025 contributor table and the independently aggregated annual total. `Other analytical units` is the arithmetic remainder; it is not a manually entered plotting value. The figure deliberately stops at analytical-unit grouping and does not infer perpetrators, targets, victim identity, or legal and historical classification.

<a id="ged-composition"></a>

<div style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 5</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">GED event composition for selected peaks</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Compute state-based, non-state and one-sided event counts for six selected peak cases retained as methodological appendix material.</p>
</div>

In [11]:
ged_cases = pd.DataFrame([
    {"case_label": "Rwanda 1994", "analysis_unit_label": "Rwanda", "year": 1994},
    {"case_label": "Syria 2014", "analysis_unit_label": "Syrian Arab Republic", "year": 2014},
    {"case_label": "Ethiopia 2022", "analysis_unit_label": "Ethiopia", "year": 2022},
    {"case_label": "Ukraine 2022", "analysis_unit_label": "Ukraine", "year": 2022},
    {"case_label": "Ukraine 2025", "analysis_unit_label": "Ukraine", "year": 2025},
    {"case_label": "Sudan 2025", "analysis_unit_label": "Sudan", "year": 2025},
])

ged_rows = []
for _, case in ged_cases.iterrows():
    row = analysis_conflict[
        (analysis_conflict["year"] == int(case["year"]))
        & (analysis_conflict["analysis_unit_label"] == case["analysis_unit_label"])
    ]
    if row.empty:
        raise AssertionError(f"Missing GED case: {case.to_dict()}")
    r = row.iloc[0]
    out = {
        "case_label": case["case_label"],
        "analysis_unit_label": case["analysis_unit_label"],
        "year": int(case["year"]),
        "ged_total_events": r["ucdp_ged_events_zf"],
        "ged_state_based_events": r["ucdp_ged_state_based_events_zf"],
        "ged_non_state_events": r["ucdp_ged_non_state_events_zf"],
        "ged_one_sided_events": r["ucdp_ged_one_sided_events_zf"],
        "ged_best_deaths": r["ucdp_ged_best_deaths_zf"],
        "ged_civilian_deaths": r["ucdp_ged_civilian_deaths_zf"],
        "ov_total_deaths": r["ucdp_ov_total_deaths_best_zf"],
        "ov_civilian_deaths": r["ucdp_ov_civilian_deaths_best_zf"],
    }
    event_sum = out["ged_state_based_events"] + out["ged_non_state_events"] + out["ged_one_sided_events"]
    out["event_type_sum_check"] = event_sum
    out["event_type_sum_matches_total"] = bool(abs(event_sum - out["ged_total_events"]) < 1e-9)
    for col in ["ged_state_based_events", "ged_non_state_events", "ged_one_sided_events"]:
        out[col.replace("events", "share")] = out[col] / out["ged_total_events"] if out["ged_total_events"] else np.nan
    ged_rows.append(out)

ged_composition = pd.DataFrame(ged_rows)
save_table(ged_composition, "ged_event_composition_selected_article_peaks.csv")
ged_composition

,case_label,analysis_unit_label,year,ged_total_events,ged_state_based_events,ged_non_state_events,ged_one_sided_events,ged_best_deaths,ged_civilian_deaths,ov_total_deaths,ov_civilian_deaths,event_type_sum_check,event_type_sum_matches_total,ged_state_based_share,ged_non_state_share,ged_one_sided_share
0,Rwanda 1994,Rwanda,1994,2119.0,57.0,0.0,2062.0,772463.0,771118.0,772463.0,771118.0,2119.0,True,0.026899,0.000000,0.973101
1,Syria 2014,Syrian Arab Republic,2014,16701.0,14253.0,1880.0,568.0,78709.0,18069.0,78709.0,18069.0,16701.0,True,0.853422,0.112568,0.034010
2,Ethiopia 2022,Ethiopia,2022,879.0,699.0,4.0,176.0,165025.0,2442.0,165025.0,2442.0,879.0,True,0.795222,0.004551,0.200228
3,Ukraine 2022,Ukraine,2022,7502.0,7292.0,0.0,210.0,100604.0,19271.0,100604.0,19271.0,7502.0,True,0.972007,0.000000,0.027993
4,Ukraine 2025,Ukraine,2025,7504.0,7496.0,0.0,8.0,90345.0,2635.0,90345.0,2635.0,7504.0,True,0.998934,0.000000,0.001066
5,Sudan 2025,Sudan,2025,718.0,510.0,3.0,205.0,75174.0,65911.0,75174.0,65911.0,718.0,True,0.710306,0.004178,0.285515


In [12]:
plot_ged = ged_composition.set_index("case_label")[[
    "ged_state_based_share", "ged_non_state_share", "ged_one_sided_share"
]].rename(columns={
    "ged_state_based_share": "state-based",
    "ged_non_state_share": "non-state",
    "ged_one_sided_share": "one-sided",
})

fig, ax = plt.subplots(figsize=FIGURE_PROFILES["wide"]["figsize"])
bottom = np.zeros(len(plot_ged))
colors = CATEGORICAL[:3]
for idx, col in enumerate(plot_ged.columns):
    ax.bar(plot_ged.index, plot_ged[col], bottom=bottom, label=col, color=colors[idx], alpha=0.86)
    bottom += plot_ged[col].values

ax.set_title("GED event composition for selected peak contributors")
ax.set_ylabel("Share of GED events")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("")
ax.set_xticklabels(plot_ged.index, rotation=25, ha="right")
ax.legend(title="GED violence type")

# Add total-event labels above bars.
for i, (_, row) in enumerate(ged_composition.iterrows()):
    ax.text(i, 1.015, f"n={fmt_int(row['ged_total_events'])}", ha="center", va="bottom", fontsize=9)

FIGURE_EXPORT_CONTEXT = {'stem':'appendix_figure_01_ged_event_composition', 'figure_id':'appendix_figure_01', 'data':ged_composition, 'metric':'Composition of GED events by violence type', 'universe':'Article-selected peak contributors with validated GED rows', 'notes':'Original stacked composition, event-count labels and legend preserved.'}
appendix_fig1_path = save_fig(
    fig,
    "appendix_fig01_ged_event_composition_selected_peaks.png",
    directory=ARTICLE_FIGURE_DIR,
)

findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font famil

### Observed result — selected cases have different recorded GED event profiles

The six selected cases have sharply different event-count compositions:

- **Rwanda 1994:** 2,119 GED events, 97.3% one-sided by event count.
- **Syria 2014:** 16,701 events, 85.3% state-based, 11.3% non-state and 3.4% one-sided.
- **Ethiopia 2022:** 879 events, 79.5% state-based and 20.0% one-sided.
- **Ukraine 2022:** 7,502 events, 97.2% state-based.
- **Ukraine 2025:** 7,504 events, 99.9% state-based.
- **Sudan 2025:** 718 events, 71.0% state-based and 28.6% one-sided.

These are **shares of recorded events**, not shares of deaths and not historical or legal conflict classifications. Event counts and fatality totals answer different questions. The appendix figure deliberately excludes Israel 2025 to avoid assigning it a public-facing violence-type label from this aggregate diagnostic alone.

<a id="intensity-diffusion"></a>

<div style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 6</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Intensity versus diffusion</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Show that aggregate intensity and number of affected units are distinct dimensions.</p>
</div>

In [13]:
diffusion_summary = annual[["year", "ucdp_ov_total_deaths", "units_with_any_ov", "units_with_ov_fatalities"]].copy()
intensity_diffusion_spearman = rank_correlation(
    diffusion_summary["units_with_ov_fatalities"],
    diffusion_summary["ucdp_ov_total_deaths"],
)
save_table(diffusion_summary, "annual_intensity_vs_diffusion.csv")

fig, ax = plt.subplots(figsize=FIGURE_PROFILES["wide"]["figsize"])
ax.scatter(diffusion_summary["units_with_ov_fatalities"], diffusion_summary["ucdp_ov_total_deaths"], s=65, color=PALETTE["blue"], alpha=0.75)
ax.set_title("Intensity and diffusion are related, but not interchangeable")
ax.set_xlabel("Analytical units with at least one best-estimate OV death")
ax.set_ylabel("Annual best-estimate OV deaths")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: fmt_int(x)))
ax.text(
    0.02, 0.95, f"Rank correlation: {intensity_diffusion_spearman:.2f}",
    transform=ax.transAxes, ha="left", va="top", fontsize=9.5, color=COLORS["muted"],
)

for year in [1994, 2014, 2022, 2025]:
    row = diffusion_summary.loc[diffusion_summary["year"] == year].iloc[0]
    ax.scatter(row["units_with_ov_fatalities"], row["ucdp_ov_total_deaths"], s=95, color=PALETTE["orange"], zorder=5)
    ax.annotate(
        str(year),
        xy=(row["units_with_ov_fatalities"], row["ucdp_ov_total_deaths"]),
        xytext=(8, 7),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
    )

FIGURE_EXPORT_CONTEXT = {
    "stem":"figure_05_intensity_vs_diffusion", "figure_id":"figure_05", "data":diffusion_summary,
    "metric":"Annual best-estimate deaths versus analytical units with positive best-estimate deaths",
    "universe":"Annual global conflict-universe aggregation, 1989–2025",
    "notes":"The x-axis uses positive fatality estimates, not the broader any-OV indicator. The descriptive rank correlation is reported; no fit or causal claim is added.",
}
fig5_path = save_fig(fig, "figure_05_intensity_vs_diffusion.png")

diffusion_summary.describe(include="all")

findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font family 'IBM Plex Sans' not found.
findfont: Font famil

,year,ucdp_ov_total_deaths,units_with_any_ov,units_with_ov_fatalities
count,37.000000,37.000000,37.000000,37.000000
mean,2007.000000,115078.135135,57.621622,55.864865
std,10.824355,136590.941803,5.913033,5.807859
min,1989.000000,21190.000000,45.000000,44.000000
25%,1998.000000,42485.000000,53.000000,51.000000
50%,2007.000000,84408.000000,59.000000,57.000000
75%,2016.000000,115556.000000,61.000000,60.000000
max,2025.000000,824177.000000,68.000000,67.000000


### Observed result — intensity and diffusion are distinct, not independent

The number of analytical units with at least one best-estimate OV death ranges from **44** to **67** per year across 1989–2025. Annual best-estimate deaths range from **21,190** to **824,177**.

The two dimensions have a **moderate positive rank association of approximately 0.58**. It would therefore be inaccurate to say that they do not move together. The defensible conclusion is narrower: they are not interchangeable, and one does not determine the other. The 1994 maximum is extreme in intensity while involving 58 fatality-positive units; 2022 has the widest fatality diffusion at 67 units and is also the second-deadliest year. By contrast, 2019 reaches 66 units but remains close to the period’s median annual toll.

<a id="manifest-validation"></a>

<div style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 7</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Export manifest and validation checks</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Write a compact summary of validated values and assert key article numbers.</p>
</div>

In [14]:
annual_sensitivity_by_estimate = annual_estimate_sensitivity.set_index("estimate")
unit_year_sensitivity_by_estimate = unit_year_estimate_sensitivity.set_index("estimate")

results_summary = {
    "panel_path": str(PANEL_PATH),
    "staging_path": str(STAGING_PATH),
    "period": "1989-2025",
    "annual_total_ov_deaths_best": float(annual_total),
    "annual_years_to_cross_50pct_low": int(annual_sensitivity_by_estimate.loc["low", "years_to_cross_50pct"]),
    "annual_years_to_cross_50pct_best": int(annual_sensitivity_by_estimate.loc["best", "years_to_cross_50pct"]),
    "annual_years_to_cross_50pct_high": int(annual_sensitivity_by_estimate.loc["high", "years_to_cross_50pct"]),
    "annual_best_years_crossing_50pct": annual_years_50["year"].astype(int).tolist(),
    "annual_best_50pct_cumulative_deaths": float(annual_years_50["ucdp_ov_total_deaths"].sum()),
    "annual_best_50pct_cumulative_share": float(annual_years_50["ucdp_ov_total_deaths"].sum() / annual_total),
    "recent_2021_2025_total_best": recent_2021_2025_total,
    "recent_2021_2025_share_best": recent_2021_2025_share,
    "recent_years_in_best_top7": recent_years_in_best_top7,
    "positive_best_unit_years": int(len(positive_unit_years)),
    "existence_bound_conflict_unit_years": int(len(analysis_conflict)),
    "unit_years_to_cross_50pct_low": int(unit_year_sensitivity_by_estimate.loc["low", "unit_years_to_cross_50pct"]),
    "unit_years_to_cross_50pct_best": int(unit_year_sensitivity_by_estimate.loc["best", "unit_years_to_cross_50pct"]),
    "unit_years_to_cross_50pct_high": int(unit_year_sensitivity_by_estimate.loc["high", "unit_years_to_cross_50pct"]),
    "diffusion_units_positive_fatalities_min": int(diffusion_summary["units_with_ov_fatalities"].min()),
    "diffusion_units_positive_fatalities_max": int(diffusion_summary["units_with_ov_fatalities"].max()),
    "intensity_diffusion_rank_correlation": float(intensity_diffusion_spearman),
    "israel_2025_rank": int(israel_2025["rank_in_year"].iloc[0]),
    "israel_2025_total_deaths_best": float(israel_2025["ucdp_ov_total_deaths_best_zf"].iloc[0]),
    "israel_2025_share_of_annual_total": float(israel_2025["share_of_annual_total"].iloc[0]),
    "article_figure4_2025_total_deaths": float(annual_total_2025),
    "article_figure4_other_units_deaths": float(other_deaths_2025),
    "article_figure4_segments": country_year_grouping_2025.to_dict(orient="records"),
}

results_path = OUTPUT_ROOT / "article_validated_results_summary.json"
results_path.write_text(json.dumps(results_summary, indent=2, ensure_ascii=False), encoding="utf-8")

figure_manifest = pd.DataFrame([
    {"figure": "Fig. 1", "description": "Annual best-estimate UCDP OV deaths, 1989–2025", "path": str(fig1_path)},
    {"figure": "Fig. 2", "description": "Best-estimate annual concentration with low/high sensitivity", "path": str(fig2_path)},
    {"figure": "Fig. 3", "description": "Selected contributor and coded civilian-share profiles", "path": str(fig3_path)},
    {"figure": "Fig. 4", "description": "2025 country-year contribution is not perpetrator attribution", "path": str(fig4_path)},
    {"figure": "Fig. 5", "description": "Best-estimate intensity versus fatality-positive diffusion", "path": str(fig5_path)},
    {"figure": "Appendix Fig. 1", "description": "GED event-count composition for selected contributors", "path": str(appendix_fig1_path)},
])
save_table(figure_manifest, "figure_manifest.csv")

# Validation checks for article-critical claims.
assert int(annual_total) == 4_257_891
assert annual_years_to_50 == 7
assert annual_years_50["year"].astype(int).tolist() == [1994, 2022, 2025, 2021, 2024, 2023, 2014]
assert int(annual_sensitivity_by_estimate.loc["low", "years_to_cross_50pct"]) == 10
assert int(annual_sensitivity_by_estimate.loc["best", "years_to_cross_50pct"]) == 7
assert int(annual_sensitivity_by_estimate.loc["high", "years_to_cross_50pct"]) == 6
assert recent_years_in_best_top7 == [2021, 2022, 2023, 2024, 2025]
assert np.isclose(recent_2021_2025_share, 0.2717420431852295)
assert len(analysis_conflict) == 9_068
assert len(positive_unit_years) == 2_067
assert unit_years_to_50 == 22
assert int(unit_year_sensitivity_by_estimate.loc["low", "unit_years_to_cross_50pct"]) == 30
assert int(unit_year_sensitivity_by_estimate.loc["best", "unit_years_to_cross_50pct"]) == 22
assert int(unit_year_sensitivity_by_estimate.loc["high", "unit_years_to_cross_50pct"]) == 17
assert int(diffusion_summary["units_with_ov_fatalities"].min()) == 44
assert int(diffusion_summary["units_with_ov_fatalities"].max()) == 67
assert np.isclose(intensity_diffusion_spearman, 0.576118, atol=1e-5)
assert int(annual.loc[annual["year"] == 1994, "ucdp_ov_total_deaths"].iloc[0]) == 824_177
assert int(annual.loc[annual["year"] == 2014, "ucdp_ov_total_deaths"].iloc[0]) == 151_743
assert int(annual.loc[annual["year"] == 2022, "ucdp_ov_total_deaths"].iloc[0]) == 317_936
assert int(annual.loc[annual["year"] == 2025, "ucdp_ov_total_deaths"].iloc[0]) == 245_464
assert int(israel_2025["rank_in_year"].iloc[0]) == 3
assert int(israel_2025["ucdp_ov_total_deaths_best_zf"].iloc[0]) == 14_563
assert np.isclose(country_year_grouping_2025["death_total"].sum(), annual_total_2025)
assert np.isclose(country_year_grouping_2025["share_of_annual_total"].sum(), 1.0, atol=1e-12)
assert fig4_path.exists(), f"Missing Article Figure 4: {fig4_path}"
assert appendix_fig1_path.exists(), f"Missing appendix GED figure: {appendix_fig1_path}"
assert figure_manifest["path"].map(lambda value: Path(value).exists()).all()
assert "Israel 2025" not in ged_cases["case_label"].tolist()

validation_table = pd.DataFrame([
    {"check": "best-estimate annual total", "status": "passed", "value": fmt_int(annual_total)},
    {"check": "annual years to 50% by estimate", "status": "passed", "value": "low 10 · best 7 · high 6"},
    {"check": "five consecutive recent years in best top seven", "status": "passed", "value": recent_years_in_best_top7},
    {"check": "2021–2025 best-estimate period share", "status": "passed", "value": fmt_pct(recent_2021_2025_share)},
    {"check": "positive / all existence-bound unit-years", "status": "passed", "value": f"{len(positive_unit_years):,} / {len(analysis_conflict):,}"},
    {"check": "unit-years to 50% by estimate", "status": "passed", "value": "low 30 · best 22 · high 17"},
    {"check": "fatality-positive diffusion range", "status": "passed", "value": "44–67"},
    {"check": "intensity–diffusion rank association", "status": "passed", "value": f"{intensity_diffusion_spearman:.2f}"},
    {"check": "Israel 2025 raw contributor metrics only", "status": "passed", "value": f"rank 3 · {fmt_int(israel_2025['ucdp_ov_total_deaths_best_zf'].iloc[0])} deaths · {fmt_pct(israel_2025['share_of_annual_total'].iloc[0])}"},
    {"check": "Article Figure 4 segments reconcile", "status": "passed", "value": f"{country_year_grouping_2025['death_total'].sum():,.0f} deaths · {country_year_grouping_2025['share_of_annual_total'].sum():.1%}"},
    {"check": "Appendix GED figure excludes Israel 2025", "status": "passed", "value": str(appendix_fig1_path)},
    {"check": "figures exported", "status": "passed", "value": len(figure_manifest)},
])
save_table(validation_table, "validation_checks.csv")

display(validation_table)
display(figure_manifest)
display_note(
    "Validation complete.",
    f"All revised article assertions passed. Summary JSON: <code>{results_path}</code>",
    kind="success",
)

,check,status,value
0,best-estimate annual total,passed,"4,257,891"
1,annual years to 50% by estimate,passed,low 10 · best 7 · high 6
2,five consecutive recent years in best top seven,passed,"[2021, 2022, 2023, 2024, 2025]"
3,2021–2025 best-estimate period share,passed,27.2%
4,positive / all existence-bound unit-years,passed,"2,067 / 9,068"
5,unit-years to 50% by estimate,passed,low 30 · best 22 · high 17
6,fatality-positive diffusion range,passed,44–67
7,intensity–diffusion rank association,passed,0.58
8,Israel 2025 raw contributor metrics only,passed,"rank 3 · 14,563 deaths · 5.9%"
9,Article Figure 4 segments reconcile,passed,"245,464 deaths · 100.0%"


,figure,description,path
0,Fig. 1,"Annual best-estimate UCDP OV deaths, 1989–2025",C:\Users\sni.digital1\Documents\avg\figures\seven-years-out-of-thirty-seven\figure_01_annual_recorded_deaths.png
1,Fig. 2,Best-estimate annual concentration with low/high sensitivity,C:\Users\sni.digital1\Documents\avg\figures\seven-years-out-of-thirty-seven\figure_02_seven_year_concentration.png
2,Fig. 3,Selected contributor and coded civilian-share profiles,C:\Users\sni.digital1\Documents\avg\figures\seven-years-out-of-thirty-seven\figure_03_peak_contributors_by_profile.png
3,Fig. 4,2025 country-year contribution is not perpetrator attribution,C:\Users\sni.digital1\Documents\avg\figures\seven-years-out-of-thirty-seven\figure_04_country_year_contribution_2025.png
4,Fig. 5,Best-estimate intensity versus fatality-positive diffusion,C:\Users\sni.digital1\Documents\avg\figures\seven-years-out-of-thirty-seven\figure_05_intensity_vs_diffusion.png
5,Appendix Fig. 1,GED event-count composition for selected contributors,C:\Users\sni.digital1\Documents\avg\figures\seven-years-out-of-thirty-seven\appendix_figure_01_ged_event_composition.png


## Final synthesis

The revised notebook validates a narrower and more defensible headline: under UCDP’s **best estimates**, seven of the 37 annual totals cross half of the 1989–2025 organized-violence death estimate. The exact threshold changes to ten years under the low estimates and six under the high estimates.

The temporal structure is not a set of evenly scattered shocks. Five consecutive years — 2021 through 2025 — belong to the best-estimate top seven and together account for 27.2% of the period total. The series therefore combines an exceptional 1994 shock, a pronounced recent cluster and substantial year-to-year variation. A single trend line is insufficient, but “no trend” is also unsupported.

At country-year grain, concentration remains strong across estimate choices, while its exact magnitude varies. The best-estimate result is 22 of 2,067 positive unit-years crossing half of the total; the low and high equivalents are 30 and 17.

Selected peak contributors also differ in measured annual contribution, coded civilian share and GED event-count composition. Those diagnostics do not independently establish historical, legal or perpetrator classifications. The safe public wording remains: **best estimate**, **positive-unit-year denominator**, **statistical contributor**, and **distinct but moderately associated intensity and diffusion**.